In [ ]:
!curl -s https://raw.githubusercontent.com/teddylee777/machine-learning/master/99-Misc/01-Colab/mecab-colab.sh | bash

--2023-11-18 12:03:16--  https://www.dropbox.com/s/9xls0tgtf3edgns/mecab-0.996-ko-0.9.2.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.2.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.2.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /s/dl/9xls0tgtf3edgns/mecab-0.996-ko-0.9.2.tar.gz [following]
--2023-11-18 12:03:17--  https://www.dropbox.com/s/dl/9xls0tgtf3edgns/mecab-0.996-ko-0.9.2.tar.gz
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc37c88f1ea4457f38e0049da420.dl.dropboxusercontent.com/cd/0/get/CHytZeDd_KAOT2nApPiqUwoFa4ljXktXG7geiVshtdxcrmaiH8RH6o0vuo-6oE3IqGNEfB3VdTXnwnb21jSVP5T3I2zYxSqODb3jTa0Ejixk7Ljmo7vr6wNV21JdWzyxyTmlRuT-wfWjlpY78u7pPpc7/file?dl=1# [following]
--2023-11-18 12:03:17--  https://uc37c88f1ea4457f38e0049da420.dl.dropboxusercontent.com/cd/0/get/CHytZeDd_KAOT2nApPiqUwoFa4ljXktXG7geiVshtdxcrmaiH8RH6o0

In [ ]:
from konlpy.tag import Okt
okt = Okt()

In [ ]:
!pip install tqdm
from tqdm import tqdm

In [ ]:
from google.colab import files
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import urllib.request
from gensim.models.word2vec import Word2Vec


In [177]:
recipe = pd.read_csv('TB_RECIPE_SEARCH-220701.csv', encoding='cp949', nrows=125000)

In [178]:
recipe

,RCP_SNO,RCP_TTL,CKG_NM,RGTR_ID,RGTR_NM,INQ_CNT,RCMM_CNT,SRAP_CNT,CKG_MTH_ACTO_NM,CKG_STA_ACTO_NM,CKG_MTRL_ACTO_NM,CKG_KND_ACTO_NM,CKG_IPDC,CKG_MTRL_CN,CKG_INBUN_NM,CKG_DODF_NM,CKG_TIME_NM,FIRST_REG_DT
0,128671,어묵김말이,어묵김말이,skfo0701,꽃날,9592,6,66,튀김,간식,가공식품류,디저트,맛있는 김말이에 쫄깃함을 더한 어묵 김말이예요-,[재료] 어묵 2개| 김밥용김 3장| 당면 1움큼| 양파 1/2개| 당근 1/2개|...,2인분,초급,60분이내,2.010000e+13
1,128892,두부에 꼬리가 달렸어요!!,NaN,skfo0701,꽃날,5538,3,26,부침,일상,해물류,밑반찬,꼬리가 너-무- 매력적인 두부새우전. 두부와 야채를 한번에!! 영양까지 만점인 두부...,[재료] 두부 1/2모| 당근 1/2개| 고추 2개| 브로콜리 1/4개| 새우 4마...,3인분,초급,30분이내,2.010000e+13
2,128932,입안에서 톡톡톡,NaN,skfo0701,꽃날,6802,8,36,굽기,일상,해물류,밥/죽/떡,간단하게 만들어 보는 알이 톡톡톡 알밥♥ 다 먹고 누룽지까지 싹싹 긁어먹는게 최고죠...,[재료] 밥 1+1/2공기| 당근 1/4개| 치자단무지 1/2개| 신김치 1쪽| 무...,2인분,초급,30분이내,2.010000e+13
3,131871,★현미호두죽,현미호두죽,cds1117,햇님&별님,2912,0,9,끓이기,일상,쌀,밥/죽/떡,현미호두죽,[재료] 현미 4컵| 찹쌀 2컵| 호두 50g| 물 1/2컵| 소금 약간,2인분,초급,30분이내,2.010000e+13
4,139247,부들부들 보들보들 북어갈비♥,북어갈비,skfo0701,꽃날,6865,3,97,굽기,술안주,건어물류,메인반찬,오늘은 집에서 굴러다니고 쉽게 구할 수 있는 북어로 일품요리를 만들어 보았어요! 도...,[재료] 북어포 1마리| 찹쌀가루 1C [양념] 간장 2T| 설탕 1T| 물 1T|...,2인분,초급,60분이내,2.010000e+13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,6948703,간단한 저녁메뉴 겨울반찬 시금치나물,시금치나물,limsu11,스마일로즈의건강밥상,1366,0,24,무침,일상,채소류,밑반찬,시금치를 좋아해서 두세 번에 나누어 씨를 뿌렸어요 농사 경험이 없어서 처음엔 모르...,[재료] 시금치 200g| 대파 흰 부분 10cm [데칠때 재료] 천일염 0.5숟가...,4인분,초급,10분이내,2.020000e+13
124996,6948704,상주곶감 곶감말이 맛있게 만드는법,곶감말,limsu11,스마일로즈의건강밥상,672,0,7,기타,명절,과일류,기타,#곶감호두말이 호두 넣고 돌돌 말아 썰어놓으면 #손님접대 나 간식으로 손색이 없는...,[재료] 곶감 10개 | 호두,4인분,초급,30분이내,2.020000e+13
124997,6948705,#굴요리 #무요리 #굴무밥만들기 #솥밥만들기 #솥밥으로 만드는 굴무밥!! #영양솥밥,굴무밥,kstencil,강철새잎,2071,0,14,기타,일상,해물류,밥/죽/떡,요즘 씨알이 굵은 굴을 많이 파는 계절입니다. 제철음식이기도 한 것이 바로 굴입니...,[재료] 생굴 50g| 무 6cm한토막| 쌀 3컵| 썬쪽파 1작은술 [양념간장재료]...,6인분이상,아무나,NaN,2.020000e+13
124998,6948706,백종원 김치찌개 저녁반찬 맛있게 만드는 법,김치찌개,limsu11,스마일로즈의건강밥상,3352,0,32,끓이기,일상,채소류,찌개,한국인들이 가장 좋아하는 #찌개 #김치찌개 된장찌개 아닌가 합니다 저도 정말 좋아...,[재료] 돼지고기 목살 250g| 신 김치1/4쪽| 물 3컵| 청양고추 1개| 홍고...,4인분,초급,30분이내,2.020000e+13


In [179]:
recipe_1 = recipe.loc[:,['CKG_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_CN', 'CKG_KND_ACTO_NM']]

In [180]:
recipe_1

,CKG_NM,CKG_STA_ACTO_NM,CKG_MTRL_CN,CKG_KND_ACTO_NM
0,어묵김말이,간식,[재료] 어묵 2개| 김밥용김 3장| 당면 1움큼| 양파 1/2개| 당근 1/2개|...,디저트
1,NaN,일상,[재료] 두부 1/2모| 당근 1/2개| 고추 2개| 브로콜리 1/4개| 새우 4마...,밑반찬
2,NaN,일상,[재료] 밥 1+1/2공기| 당근 1/4개| 치자단무지 1/2개| 신김치 1쪽| 무...,밥/죽/떡
3,현미호두죽,일상,[재료] 현미 4컵| 찹쌀 2컵| 호두 50g| 물 1/2컵| 소금 약간,밥/죽/떡
4,북어갈비,술안주,[재료] 북어포 1마리| 찹쌀가루 1C [양념] 간장 2T| 설탕 1T| 물 1T|...,메인반찬
...,...,...,...,...
124995,시금치나물,일상,[재료] 시금치 200g| 대파 흰 부분 10cm [데칠때 재료] 천일염 0.5숟가...,밑반찬
124996,곶감말,명절,[재료] 곶감 10개 | 호두,기타
124997,굴무밥,일상,[재료] 생굴 50g| 무 6cm한토막| 쌀 3컵| 썬쪽파 1작은술 [양념간장재료]...,밥/죽/떡
124998,김치찌개,일상,[재료] 돼지고기 목살 250g| 신 김치1/4쪽| 물 3컵| 청양고추 1개| 홍고...,찌개


In [181]:
recipe_1[recipe_1['CKG_NM'].isnull()]

,CKG_NM,CKG_STA_ACTO_NM,CKG_MTRL_CN,CKG_KND_ACTO_NM
1,NaN,일상,[재료] 두부 1/2모| 당근 1/2개| 고추 2개| 브로콜리 1/4개| 새우 4마...,밑반찬
2,NaN,일상,[재료] 밥 1+1/2공기| 당근 1/4개| 치자단무지 1/2개| 신김치 1쪽| 무...,밥/죽/떡
34,NaN,술안주,[재료] 닭날개 20개| 재움용 우유 1컵 [허니머스타드소스] 머스타드 1큰술| 마...,양식
50,NaN,간식,NaN,빵
65,NaN,손님접대,[재료] 한우 우둔살 400그램| 파프리카 1개| 피망 1개| 양파 큰것 1/2개|...,양식
...,...,...,...,...
123405,NaN,일상,[재료] 감자 2개| 무 1조각| 보리새우 1줌| 팽이버섯 1덩어리| 양파 1/5개...,국/탕
123406,NaN,간식,[재료] 라이스페이퍼 8장| 햄 100g| 슬라이스치즈 2장| 양파 1/2개| 계란...,퓨전
124529,NaN,일상,[재료] 중화펜 1 [양념] 돼지기름 1국자| 식용유 1국자,기타
124639,NaN,일상,[가자미 구이재료] 가자미 3마리| 식용유 약간 [무수분 돼지고기 수육재료] 돼지고...,메인반찬


In [182]:
recipe_2 = recipe_1.drop_duplicates(['CKG_NM'])

In [183]:
recipe_2

,CKG_NM,CKG_STA_ACTO_NM,CKG_MTRL_CN,CKG_KND_ACTO_NM
0,어묵김말이,간식,[재료] 어묵 2개| 김밥용김 3장| 당면 1움큼| 양파 1/2개| 당근 1/2개|...,디저트
1,NaN,일상,[재료] 두부 1/2모| 당근 1/2개| 고추 2개| 브로콜리 1/4개| 새우 4마...,밑반찬
3,현미호두죽,일상,[재료] 현미 4컵| 찹쌀 2컵| 호두 50g| 물 1/2컵| 소금 약간,밥/죽/떡
4,북어갈비,술안주,[재료] 북어포 1마리| 찹쌀가루 1C [양념] 간장 2T| 설탕 1T| 물 1T|...,메인반찬
5,토마토스파게티,일상,[재료] 파스타면 [양념] 토마토 1개| 토마토 페이스트 3T| 양파 1/2개| 다...,면/만두
...,...,...,...,...
124985,모짜치즈고구마,초스피드,[재료] 고구마 1개| 모짜렐라치즈| 꿀 맘대로,디저트
124987,크리스마스초코케이크,간식,[재료(1호 사이즈)] 1호 사이즈 초코 케이크 시트 3장| 달걀 흰자 5개_ 17...,디저트
124990,돼지김치전,술안주,[재료] 김치 1/2포기| 양파 1/2개| 애호박 1/4개| 돼지목살 200g [양...,기타
124992,검은콩바나나두부쉐이크,다이어트,[재료] 우유 200ml| 두부 1/2모| 바나나 1개| 삶은 검은콩 3t,기타


In [184]:
recipe_3 = recipe_2.dropna(subset = ['CKG_NM'])
recipe_3

,CKG_NM,CKG_STA_ACTO_NM,CKG_MTRL_CN,CKG_KND_ACTO_NM
0,어묵김말이,간식,[재료] 어묵 2개| 김밥용김 3장| 당면 1움큼| 양파 1/2개| 당근 1/2개|...,디저트
3,현미호두죽,일상,[재료] 현미 4컵| 찹쌀 2컵| 호두 50g| 물 1/2컵| 소금 약간,밥/죽/떡
4,북어갈비,술안주,[재료] 북어포 1마리| 찹쌀가루 1C [양념] 간장 2T| 설탕 1T| 물 1T|...,메인반찬
5,토마토스파게티,일상,[재료] 파스타면 [양념] 토마토 1개| 토마토 페이스트 3T| 양파 1/2개| 다...,면/만두
6,표고버섯탕수,손님접대,[재료] 건표고버섯 9개| 오이 1/2개| 당근 1/2개| 양파 1/2개| 사과 1...,메인반찬
...,...,...,...,...
124985,모짜치즈고구마,초스피드,[재료] 고구마 1개| 모짜렐라치즈| 꿀 맘대로,디저트
124987,크리스마스초코케이크,간식,[재료(1호 사이즈)] 1호 사이즈 초코 케이크 시트 3장| 달걀 흰자 5개_ 17...,디저트
124990,돼지김치전,술안주,[재료] 김치 1/2포기| 양파 1/2개| 애호박 1/4개| 돼지목살 200g [양...,기타
124992,검은콩바나나두부쉐이크,다이어트,[재료] 우유 200ml| 두부 1/2모| 바나나 1개| 삶은 검은콩 3t,기타


In [431]:
recipe_4 = recipe_3.dropna(subset = ['CKG_MTRL_CN'])
recipe_4

,CKG_NM,CKG_STA_ACTO_NM,CKG_MTRL_CN,CKG_KND_ACTO_NM
0,어묵김말이,간식,[재료] 어묵 2개| 김밥용김 3장| 당면 1움큼| 양파 1/2개| 당근 1/2개|...,디저트
3,현미호두죽,일상,[재료] 현미 4컵| 찹쌀 2컵| 호두 50g| 물 1/2컵| 소금 약간,밥/죽/떡
4,북어갈비,술안주,[재료] 북어포 1마리| 찹쌀가루 1C [양념] 간장 2T| 설탕 1T| 물 1T|...,메인반찬
5,토마토스파게티,일상,[재료] 파스타면 [양념] 토마토 1개| 토마토 페이스트 3T| 양파 1/2개| 다...,면/만두
6,표고버섯탕수,손님접대,[재료] 건표고버섯 9개| 오이 1/2개| 당근 1/2개| 양파 1/2개| 사과 1...,메인반찬
...,...,...,...,...
124985,모짜치즈고구마,초스피드,[재료] 고구마 1개| 모짜렐라치즈| 꿀 맘대로,디저트
124987,크리스마스초코케이크,간식,[재료(1호 사이즈)] 1호 사이즈 초코 케이크 시트 3장| 달걀 흰자 5개_ 17...,디저트
124990,돼지김치전,술안주,[재료] 김치 1/2포기| 양파 1/2개| 애호박 1/4개| 돼지목살 200g [양...,기타
124992,검은콩바나나두부쉐이크,다이어트,[재료] 우유 200ml| 두부 1/2모| 바나나 1개| 삶은 검은콩 3t,기타


No charts were generated by quickchart


In [186]:
print(len(recipe_4))

37169


In [432]:
# 불용어 정의
stopwords = ['|']

# 형태소 분석기 OKT를 사용한 토큰화 작업 (다소 시간 소요)
okt = Okt()

tokenized_data = []
for sentence in tqdm(recipe_4['CKG_MTRL_CN']):
    tokenized_sentence = okt.morphs(sentence, stem=True) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    tokenized_data.append(stopwords_removed_sentence)

100%|██████████| 37169/37169 [02:42<00:00, 228.36it/s]


In [433]:
print(tokenized_data[:100])

[['[', '재료', ']', '어묵', '2', '개', '김밥', '용김', '3', '장', '당면', '1', '움큼', '양파', '1/2', '개', '당근', '1/2', '개', '깻잎', '6', '장', '튀김', '가루', '1', '컵', '올리브유', '적당', '량', '간장', '1', 'T', '참기름', '1', 'T'], ['[', '재료', ']', '현미', '4', '컵', '찹쌀', '2', '컵', '호두', '50', 'g', '물', '1/2', '컵', '소금', '약간'], ['[', '재료', ']', '북어', '포', '1', '마리', '찹쌀', '가루', '1', 'C', '[', '양념', ']', '간장', '2', 'T', '설탕', '1', 'T', '물', '1', 'T', '다지다', '파', '1', 'T', '다지다', '마늘', '1', 'T', '참기름', '1', 'T', '깨소금', '1', 'T', '후춧가루', '약간'], ['[', '재료', ']', '파스타', '면', '[', '양념', ']', '토마토', '1', '개', '토마토', '페이스', '트', '3', 'T', '양파', '1/2', '개', '다지다', '마늘', '1', 'T', '피망', '1/2', '개', '올리브유', '3', 'T'], ['[', '재료', ']', '건', '표고버섯', '9', '개', '오이', '1/2', '개', '당근', '1/2', '개', '양파', '1/2', '개', '사과', '1/2', '쪽', '그', '외', '의', '야채', '과일', '[', '녹말', '물', ']', '녹말가루', '2', 'C', '물', '1', 'C', '계란', '노른자', '1', '개', '[', '탕수', '소스', ']', '물', '2', 'C', '설탕', '1/2', 'C', '식초', '3', 'T', '간장', '1', 'T', '녹말', '물', '2'

In [434]:
from konlpy.tag import Okt

# 불용어 정의
stop_words = set("[ ] | / \n".split(' '))

# Okt 객체 생성
okt = Okt()

# CKG_NM 칼럼에 대해 불용어 제거 수행
recipe_4['CKG_MTRL_CN'] = recipe_4['CKG_MTRL_CN'].apply(lambda x: ' '.join([word for word in okt.morphs(x) if word not in stop_words]))

# 불용어 제거 후 CKG_NM 칼럼 출력
print(recipe_4['CKG_MTRL_CN'])

0         재료 어묵 2 개 김밥 용김 3 장 당면 1 움큼 양파 1/2 개 당근 1/2 개 ...
3                    재료 현미 4 컵 찹쌀 2 컵 호두 50 g 물 1/2 컵 소금 약간
4         재료 북어 포 1 마리 찹쌀 가루 1 C 양념 간장 2 T 설탕 1 T 물 1 T ...
5         재료 파스타 면 양념 토마토 1 개 토마토 페이스 트 3 T 양파 1/2 개 다진 ...
6         재료 건 표고버섯 9 개 오이 1/2 개 당근 1/2 개 양파 1/2 개 사과 1/...
                                ...                        
124985                             재료 고구마 1 개 모짜렐라 치즈 꿀 맘대로
124987    재료 ( 1 호 사이즈 )] 1 호 사이즈 초코 케이크 시트 3 장 달걀 흰자 5 ...
124990    재료 김치 1/2 포기 양파 1/2 개 애호박 1/4 개 돼지 목살 200 g 양념...
124992            재료 우유 200 ml 두부 1/2 모 바나나 1 개 삶은 검은 콩 3 t
124996                                        재료 곶감 10 개 호두
Name: CKG_MTRL_CN, Length: 37169, dtype: object


In [190]:
from gensim.models import Word2Vec
model = Word2Vec(sentences = tokenized_data, window = 5, min_count = 5, workers = 4, sg = 0)
print('완성된 임베딩 매트릭스의 크기 확인 :', model.wv.vectors.shape)

완성된 임베딩 매트릭스의 크기 확인 : (3194, 100)


<<모델 학습>>

In [191]:
sentences = [recipe.split() for recipe in recipe_4['CKG_MTRL_CN']]
model = Word2Vec(sentences, vector_size=200, window=8, min_count=1, workers=4)

In [192]:
model = Word2Vec(sentences, vector_size=128, window=5, min_count=5, workers=4, sg=1)

In [193]:
model = Word2Vec(sentences, vector_size=300, window=5, min_count=1, workers=4, sg=1)

In [194]:
model = Word2Vec(sentences, vector_size=128, window=10, min_count=1, workers=4, sg=1)

In [195]:
model = Word2Vec(sentences, vector_size=128, window=5, min_count=1, workers=4, sg=1)

In [196]:
from gensim.models import Word2Vec

# sentences: 학습 데이터 (리스트 형태의 토큰화된 문장들)
# vector_size: 단어 벡터의 차원 수
# window: 문맥 윈도우 크기
# min_count: 최소 단어 빈도
# workers: 학습에 사용되는 CPU 스레드 수
# sg: Skip-gram 모델 사용 여부 (sg=1: 사용, sg=0: CBOW 모델 사용)
# negative: 부정적 예제의 수
# epochs: 학습 반복 횟수
model = Word2Vec(sentences, vector_size=128, window=5, min_count=5, workers=4, sg=1, negative=5, epochs=100)

In [197]:
print(model.wv.most_similar("어묵"))

[('사각', 0.7146353721618652), ('오뎅', 0.6806772351264954), ('구멍', 0.5320094227790833), ('부산', 0.5298781394958496), ('뚫린', 0.5258014798164368), ('깻잎', 0.5036743879318237), ('당면', 0.4960436522960663), ('떡', 0.4817240536212921), ('납작', 0.47658029198646545), ('유부', 0.4734435975551605)]


In [586]:
import pandas as pd
import re
from konlpy.tag import Okt

data = recipe_4

# 예외 목록 정의
exceptions = ["된장", "고추장", "참기름", "간장", "모차렐라치즈"]

# 특정 단어 조합 목록 정의
word_combinations = {
    "튀김 가루" : "튀김가루", "북어 포": "북어포", "찹쌀 가루": "찹쌀가루","크리스탈 고운 소금":"소금","닭다리 살":"닭다리살", "파스타 면" : "파스타면", "페이스 트" : "페이스트", "건 표고버섯" : "건표고버섯", "녹말 물" : "녹말물","탕수 소스":"탕수소스","다진 당근":"다진당근","다진 피망":"다진피망","오 겹살":"오겹살","왕만 두":"왕만두","치 커리":"치커리",
    "마늘 솔트":"마늘솔트", "샐러드 드레싱":"샐러드드레싱","레몬 즙" : "레몬즙","통 마늘":"통마늘","다진 대파":"다진대파","다진 마늘":"다진마늘","다진 생강":"다진생강","다진 파":"다진파","용김":"용 김","닭 양념":"닭양념","다진 김치":"다진김치","파 뿌리":"파뿌리","부추 부추 무침":"부추",
    "홍합 살":"홍합살","후추 가루": "후추가루","홀 레인 머스터드":"홀레인머스타드","허니 머스터드":"허니머스타드","다진 양파":"다진양파","체다 치즈":"체다치즈","노란 치즈":"노란치즈","모차렐라 치즈":"모짜렐라치즈","풋 고추":"풋고추","봄 동 잎":"봄동잎","홍차 티백":"홍차티백","군 두":"군만두",
    "청 고추":"청고추","홍 고추":"홍고추","밥 쌀":"밥","모닝 빵":"빵","모짜렐라 치즈":"모짜렐라치즈","커스타드 크림":"커스타드크림","바닐라 오일":"바닐라오일","쌀 가루":"쌀가루","드라이 이스트":"드라이이스트","게 조개":"게조개","계란 노른자":"노른자","카 놀라 유":"카놀라유","꼬막 육수":"꼬막육수",
    "쿠키 반죽":"쿠키반죽","소금 소금":"소금","청 홍고추":"청고추 홍고추","오코노미야키 반죽":"오코노미야키반죽","혼다 시":"혼다시","돈가스 소스":"돈가스소스","만 두피":"만두피","포도 씨유":"포도씨유","제노아 즈":"제노아즈","마늘 대파 소스":"","들깨 가루":"들깨가루","사과 쨈":"사과쩀",
    "꽈리 고추":"꽈리고추","요리 당":"요리당","양념장":"[양념]","양념":"[양념]","재료":"[재료]","월계수 잎":"월계수잎","생강 가루":"생강가루","겨자 가루":"겨자가루","용어 묵":"어묵","캔햄 이나 용햄":"햄","홍시 대 봉시":"홍시 대봉시","매실 엑기스":"매실엑기스","김 가루":"김가루","사골 육수":"사골육수",
    "호박 씨":"호박씨","슈가 파우더":"슈가파우더","단호박 가루":"단호박가루","시나몬 가루":"시나몬가루","반 건조 오징어":"반건조오징어","계란 흰자":"계란흰자","박력":"박력분","박력분분":"박력분","강력":"강력분","베이 킹 파우더":"베이킹파우더","베이 킹 소 다":"베이킹소다","초코 칩":"초코칩","초코 뮈슬리":"초코뮈슬리","두부 면":"두부면",
    "아몬드 크림":"아몬드크림","크림 치즈":"크림치즈","아몬드 가루":"아몬드가루","항정 살이 등심 덧살":"항정살","코코넛 가루":"코코넛가루","플레인 요거트":"플레인요거트","강력분분":"강력분","인 스턴 드":"인스턴트","헤비 위핑크림":"헤비위핑크림","딸기 젤리 [재료]":"[재료]","새 송이버섯":"새송이버섯","국 물":"국물","우동 면":"우동면",
    "마늘 다진것":"다진마늘","밤 페이스트":"밤페이스트","밤 베이킹파우더 계란 우유 밤":"베이킹파우더 계란 우유 밤","건 고추":"건고추","닭 밑":"[닭 밑간]","생강 즙":"생강즙","캬라멜 소스":"카라멜소스","두 툼 하 게":"두툼하게","판 젤라틴":"판젤라틴","생 고등어":"고등어","검은 콩":"검은콩","멸치 육수":"멸치육수",
    "브 리 오슈":"브리오슈","슈거 파우더":"슈가파우더","듬 해물":"모듬해물","양조 간장":"양조간장","실액":"매실액","칵테일 새우":"칵테일새우","육수 새우 머리":"새우머리","피자 피즈":"피자치즈","엔젤 헤어 파스타":"엔젤헤어파스타","시 칠 미":"시칠미","코코아 가루":"코코아가루","바닐라 액":"바닐라액","양파 즙":"양파즙","니 신":"니신","정사 호팬 분량 시트":"",
    "다크 초콜릿":"다크초콜릿","바질 잎":"바질잎","파 마산 치즈":"파마산치즈","올리브유 물 드라이이스트 올리브유":"올리브유","흑 미 가루":"흑미가루","후 랑 크 소세지":"후랑크소세지","진 간장":"진간장","꽃 소금":"꽃소금","딸기 퓨레":"딸기퓨레","미로 와":"미로와","바닐라 엑스 트렉":"바닐라엑스트렉",
    "다시 마물":"다시마물","고추 가루":"고춧가루","튀김 옷":"튀김옷","고운 고춧가루":"고춧가루","다시마 국물":"다시마국물","밥 이랑 밥 뿌려 먹는 가루":"밥에뿌려먹는가루","다시마 육수":"다시마육수","닭 가슴 살캔":"닭가슴살","청 피망":"청피방","홍 피망":"홍피망","완두 콘캔":"완두콩","커스터드 크림 [재료]":"[재료]","샌드 재료":"","닭 안심 살":"닭안심",
    "닭고기 안심 살":"닭고기안심","바베큐 소스":"바베큐소스","오렌지 쥬스":"오렌지주스","식빵 소스":"[식빵 소스]","베이비 채소":"베이비채소","허브 솔트":"허브솔트","호떡 믹스":"호떡믹스","카야 잼":"카야잼","녹차 가루":"녹차가루","딸기 쥬스 가루":"딸기쥬스가루","자숙 문어":"자숙문어","치킨 스톡":"치킨스톡",
    "훈제 닭 가슴 살":"훈제닭가슴살","해바라기 유":"해바라기유","새 조개":"새조개","피 가 얇은 냉동 고기":"","건무 화 과":"무화과","홍 이 장군":"홍이장군","군 고구마":"군고구마","꼬치 소스":"","만두 물만두 고기만 두 물만두 일 때":"만두","스 위트 칠리":"스위트칠리","스프 봉":"","떡국 떡":"떡","묵은 김치":"묵은김치","생 이":"매생이",
    "참치 완 자":"참치완자","적 고구마":"적고구마","국 그릇":"국그릇","따듯 한 물":"따듯한물","다크 쵸콜릿":"다크초콜릿","초코 펜 분홍":"초코펜","부침 가루":"부침가루","늙은 호박":"늙은호박","신 김치":"신김치","국 간장":"국간장","가루 한천":"가루한천","짜장 분말":"짜장분말","반 건조 가자미":"반건조가자미",
    "매실 원 액":"매실원액","두 반장":"두반장","느타리 비섯":"느타리버섯","탕 수 소스":"탕수소스","크 래미":"크래미","카레 가루":"카레가루","왕만두 일반 만두":"만두","김치 고기만 두":"김치만두 고기만두","군 만두":"군만두","민트 잎":"민트잎","파슬리 가루":"파슬리가루","캔햄":"햄","샐러 리":"샐러리",
    "달걀 지단":"달걀지단","다진 돼지고기":"다진돼지고기","다진소고기":"다진소고기","로스 팜 런 천 미트":"","가 쓰 오브 시":"가쓰오부시","백 설탕":"백설탕", "편 마늘":"편마늘","고기만 두":"고기만두","김치 만두":"김치만두","리 챔":"리챔","우리 밀 백 밀가루 중력":"밀가루중력분","후르 츠캔":"후르츠캔","연어 통살캔":"연어캔","노랑 파프리카 파프리카":"노랑파프리카","허니 두 캔더 로프":"허니두캔더로프"}

# 불용어 목록 정의
stopwords = ["컵", "T", "약간", "맘대로", "t", "CC","cc","제일제당","동원","ea","Real","발아현미", "C"," 만들기","동", "s","oz","-","CJ","cm", "L", "등","아빠숟가락","채소", "개", "장", "움큼", "장", "g","먹기", "좋게",  "마리", "쪽", "그외의", "중간짜리", "조금", "S", "or", "담백한","미니","씩","TS",
             "매콤","한","사정", "대로" ,"준비", "하세요","시판","잘","익은","수저","중간크기", "or","이나", "또는", "포기", "중간","큰크기","적당히", "모두가능", "적당", "간","하림","기","데친", "량", "모", "큰", "술", "굵은","토", "핑할", "ml", "김밥", "깍","밑둥", "둑", "썬", "염장", "썬것", "줌", "다듬은",
             "것", "분","스푼","매","중","냉장고","머","든","상관없답니다","향신료","동전굵만큼", "봉지", "kg", "삶은", "양", "ts", "톡톡", "도막","자투리","일반","냉동","백설","다담","캔","그","갓","지은", "외", "의", "야채","용","공기","크게","큰거","떡볶이","통","싱글","팩","샌드위치","작은",
             "조미","걸","로","골","라서","나","조","각","여","불린","토막","한코","찜용","대체","가능","외쌈","알","정도","지을","백세","으","깬것","커스타드크림","국내","산","제빵","그램","인스턴트","조림","좋아하는","종류","줄","오코노미야키반죽","손","줄기","대략","큰것","유기농","잘게","자른",
             "마른","빨강","몸통","만", "사용", "송송", "썬잔","둠","묶음","전장","인","set","G","타르트","지","Ts","각종","당","절임","토핑용","속","에","드실만큼","넣었던","목","안팎","얼린것","고급", "우촌", "생생","숙성","건","이내", "된","ABCT", "남은것","판","종이컵","인분","아무","약","부위",
             "크기","장식","TBSP","꼬집","데코","레이","션","이상","줄줄이","대","사이즈","포","나박","맛","튀김옷","고","밑","껍질","깐","작은것","숭가락","넉넉히","이","찜","tsp","치밥치떡","금기","숟가락","세트","남는","사리","원","클래식","인치", "동전","군","색깔","별로","취향", "껏","저","지방","개정",
             "도","봉","뜯기전","선택","먹고","남은","깉은","생략","mL","많이","삶", "을", "때", "미","샘표","맛있어지는","갈은","고명","갈은","cup","조리", "도구", "나무꼬","기호","봄","숟갈","원칙", "지키는"]

def remove_unwanted_characters(text):
    if text in exceptions:
        return text
    text = re.sub(r'[^\w\s]', '', text)  # 특수 문자 제거
    text = re.sub(r'\d+', '', text)  # 숫자 제거
    text = re.sub(r'[\u2E80-\u2FD5\u3190-\u319f\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]', '', text)  # 한자 제거
    return text.strip()

def remove_spaces_for_specific_combinations(text):
    for combo in word_combinations:
        if combo in text:
            text = text.replace(combo, word_combinations[combo])
    return text

def clean_ingredient(ingredient):
    # 예외 목록에 있는 경우, 그대로 반환
    if ingredient in exceptions:
        return ingredient

    ingredient = remove_unwanted_characters(ingredient)
    ingredient = remove_spaces_for_specific_combinations(ingredient)

    # 각 단어별로 불용어 제거
    words = ingredient.split()
    cleaned_words = [word for word in words if word not in stopwords]

    # 정제된 단어들을 다시 합침
    combined_word = ' '.join(cleaned_words)
    if combined_word in exceptions:
        return combined_word

    return combined_word

# 데이터 처리를 수행합니다.
recipe_4['CKG_MTRL_CN'] = recipe_4['CKG_MTRL_CN'].apply(
    lambda x: clean_ingredient(str(x)) if pd.notnull(x) else x
)

# 수정된 코드의 결과를 확인하기 위해 상위 5개 결과를 출력
print(recipe_4['CKG_MTRL_CN'].iloc[23565])
name = recipe_4['CKG_NM'].iloc[23565]
name

[재료] 밥 치킨 양파 계란 [양념] 간장 맛술 설탕


'치킨가라아게동'

In [587]:
similar_words = model.wv.most_similar("참치", topn=10)

# 유사한 단어와 연관된 메뉴명 찾기
for word, similarity in similar_words:
    # 연관된 메뉴명을 찾는 코드를 추가하면 됨
    similar_menus = recipe_4[recipe_4['CKG_MTRL_CN'].str.contains(word, case=False)]
    print(f"Similar Word: {word}, Similarity: {similarity}")
    print("Associated Menus:")
    print(similar_menus[['CKG_NM', 'CKG_MTRL_CN']])
    print("\n")

Similar Word: 캔, Similarity: 0.5942673087120056
Associated Menus:
           CKG_NM                                        CKG_MTRL_CN
1319        후르츠머핀              [재료] 버터 설탕 계란 박력분 베이킹파우더 우유 캔디 드 믹스 필
2019           파이         [재료] 밀가루중력분 레이크 소금 차가운 버터 계란 유정 란 찬물 하드 캔디
5355      닭다리살라자냐  [재료] 닭다리살캔 라자냐 파스타 우유 파슬리가루 피자치즈 [양념] 스파게티 소스 ...
9159        북어포무침                        [재료] 골벵이캔 북어포 소면 초고추장 본문 참조
16949   오미자후르츠팥빙수                      [재료] 오미자 청 물 후르츠캔 빙수 팥 연유 시리얼
17093      복숭아모히또               [재료] 탄산수 민트잎 레몬즙 황도 복숭아 생 블루베리 캔즙 얼음
18568        호박약식  [재료] 찹쌀 백 설탕 소금 물 호두 잣 린밤 캔밤 대추 늙은호박고지 [양념] 단호...
25597      연어야채밥전  [재료] 연어캔 밥 양파 청피방 당근 청양고추 계란 밀가루 빵가루 [양념] 소금 후...
26721    베이컨갈릭콘치즈  [재료] 옥수수 콘캔 양파 노란 파프리카 빨간 파프리카 모짜렐라치즈 기호 베이컨 편...
37885   돌나물골뱅이초무침  [재료] 골뱅이 돌나물 양배추 [양념] 고추장 고춧가루 올리고당 골뱅이 캔속 국물 ...
37941    주꾸미라면파스타  [재료] 주꾸미 라면 봄 적양배추 크림 파스타 소스 병 캔커피 [양념] 버터 다진마...
43007      후루츠사라드                      [재료] 허니두캔더로프 수박 포도 파인애플 체리 딸기
54955     옥수수치즈식빵                    

In [588]:
# Set the words for which you want to find similar recipes
words_to_find = ["김치", "두부"]

# Find similar words for each individual word
similar_words = [model.wv.most_similar(word, topn=5) for word in words_to_find]

# Flatten the list of similar words into a set of unique words
unique_words = set(word for sublist in similar_words for word, _ in sublist)

# Create a combined mask for the unique words
combined_mask = recipe_4['CKG_MTRL_CN'].apply(lambda x: all(word in x for word in unique_words))

# Filter the DataFrame based on the combined mask
filtered_recipes = recipe_4[combined_mask]

# Print the sorted recipes
for _, similarity in sorted(similar_words[0], key=lambda x: x[1], reverse=True):
    print(f"\nSimilarity: {similarity}")

    # Print associated menus
    associated_menus = filtered_recipes[filtered_recipes['CKG_MTRL_CN'].str.contains(_, case=False)]

    if not associated_menus.empty:
        print("Associated Menus:")
        print(associated_menus[['CKG_NM', 'CKG_MTRL_CN']])
    else:
        print("No associated menus found.")


Similarity: 0.7069230079650879
No associated menus found.

Similarity: 0.6956360340118408
No associated menus found.

Similarity: 0.6858661770820618
No associated menus found.

Similarity: 0.6279127597808838
No associated menus found.

Similarity: 0.6073418855667114
No associated menus found.


In [589]:
import re

# Function to escape special characters in a string
def escape_special_characters(text):
    special_characters = r"[\^$.|?*+(){}"
    return re.sub(f"[{re.escape(special_characters)}]", r"\\\g<0>", text)

# Example usage:
similar_words = model.wv.most_similar("김치", "간장", topn=5)

# Iterate through similar words and find associated menus
for word, similarity in similar_words:
    # Escape special characters in the word
    escaped_word = escape_special_characters(word)

    # Find associated menus using the escaped word
    similar_menus = recipe_4[recipe_4['CKG_MTRL_CN'].str.contains(escaped_word, case=False)]

    print(f"Similar Word: {word}, Similarity: {similarity}")
    print("Associated Menus:")
    print(similar_menus[['CKG_NM', 'CKG_MTRL_CN']])
    print("\n")

Similar Word: 배추김치, Similarity: 0.42272642254829407
Associated Menus:
               CKG_NM                                        CKG_MTRL_CN
48      소고기크림치즈오븐스파게티      [재료] 배추김치 소고기 체다치즈 스파게티 국수 모짜렐라치즈 [양념] 헤비위핑크림
1077      반건조오징어김치부침개                    [재료] 부침가루 물 배추김치 반건조오징어 어묵 카놀라유
1352           김치만두전골  [재료] 배추김치 돼지고기 만두 파 실파뿌리 팽이버섯 양파 [양념] 다진마늘 간장 ...
2099         참치완자김치찌개  [재료] 배추김치 물 김칫국물 풋고추 대파 식용유 참치완자 참치 두부 양파 밀가루 ...
2320            햄김치찌개   [재료] 배추김치 물 멸치 김칫국물 양파 햄 [양념] 고추장 고춧가루 다진마늘 후춧가루
...               ...                                                ...
121027       열무김치참치볶음  [재료] 열무김치 배추김치 고추 참치 순 살 참치 들기름 다진마늘 매매매매매실액 고...
121040     훈제닭가슴살김치볶음  [재료] 배추김치 훈제닭가슴살 [양념] 진간장 다진마늘 해바라기유 고춧가루 올리고당...
122340        씨앗젓갈주먹밥        [재료] 밥 씨앗 젓갈 김가루 참기름 통깨 설탕 들기름 고춧가루 통깨 배추김치
123310        베이컨김치볶음                 [재료] 배추김치 베이컨 다진파 [양념] 버터 오일 설탕 통깨
124258      새송이버섯김치볶음  주 [재료] 배추김치 혹은 막 김치 국그릇 새송이버섯 [양념] 올리브오일 들깨가루 ...

[171 rows x 2 columns]


Similar Word: 묵은지, Similarit

In [ ]:
similar_words_kimchi_tofu = model.wv.most_similar("고기", "두부", topn=5)

# Get unique words from similar_words
unique_words = set(word for word, _ in similar_words_kimchi_tofu)

# Create a combined mask for the unique words
combined_mask = recipe_4['CKG_MTRL_CN'].apply(lambda x: all(word in x for word in unique_words))

# Filter the DataFrame based on the combined mask
filtered_recipes = recipe_4[combined_mask]

# Print the sorted recipes
for index, (word, similarity) in enumerate(similar_words_kimchi_tofu):
    print(f"\nSimilar Words: {word}, Similarity: {similarity}")

# Print associated menus
if not filtered_recipes.empty:
    print("\nAssociated Menus:")
    print(filtered_recipes[['CKG_NM', 'CKG_MTRL_CN']])
else:
    print("No associated menus found.")


Similar Words: 푸실리/펜네, Similarity: 0.7044182419776917

Similar Words: 3개씩|, Similarity: 0.6925780773162842

Similar Words: 오가닉, Similarity: 0.6908244490623474

Similar Words: [데코], Similarity: 0.687985897064209

Similar Words: [쉘(마카롱, Similarity: 0.6871306896209717
No associated menus found.


In [ ]:
similar_words = model.wv.most_similar("삼겹살", topn=5)

# 유사한 단어와 연관된 메뉴명 찾기
for word, similarity in similar_words:
    # 연관된 메뉴명을 찾는 코드를 추가하면 됨
    similar_menus = recipe_4[recipe_4['CKG_MTRL_CN'].str.contains(word, case=False)]

    # Display only the menu names
    similar_menu_names = similar_menus['CKG_NM'].unique()

    print(f"Similar Word: {word}, Similarity: {similarity}")
    print("Associated Menus:")
    print(similar_menu_names)
    print("\n")


Similar Word: 목살, Similarity: 0.9551011919975281
Associated Menus:
['부추제육볶음' '돼지고기굴소스볶음' '제육볶음' '돼지목살양념구이' '돼지고기두루치기' '떡볶이제육볶음' '돼지고기볶음우동'
 '돼지고추장불고기' '고추장불고기볶음우동' '된장목살구이' '목살스테이크' '폭찹' '돼지목살김치찜' '돼지고기볶음'
 '돼지고기김치두루치기' '비지찌개' '돼지목살된장구이' '대구콩나물찜' '돼지고기맥적' '돼지고기콩비지찌개' '돼지목살두부김치찌개'
 '돼지목살수육' '고추잡채호빵' '달래돼지고기된장구이' '돼지고기카레볶음' '문어동파육' '콩삼불고기' '돼지고기청국장'
 '떡볶이떡사과불고기' '돼지고기생강구이' '돼지갈비구이' '돼지두루치기' '더덕제육볶음' '느타리버섯간장불고기' '목살폭찹'
 '오징어돼지불고기덮밥' '맥적' '깐풍육' '우거지찜' '분짜' '목살폭찹스테이크' '돼지고기두반장볶음' '돼지고기냉채'
 '돼지고기된장구이' '돼지고기된장볶음' '돼지고기고추장감자찌개' '돼지고기오징어볶음' '두릅돼지고기볶음' '간장보쌈'
 '돼지고기마늘구이' '돼지고기목살김치찌개' '삼겹살된장구이' '돼지목살생강조림롤' '봄동돼지고기고추장찌개' '수육' '목살구이'
 '돼지고기감자고추장볶음' '돼지고기야채조림' '돈지루' '맥적구이' '목살찜' '돼지고기제육볶음' '돼지고기맥적구이'
 '부추돼지고기볶음밥' '목살김치찌개' '미나리떡제육볶음' '필라프' '김치돼지고기볶음' '카레목살스테이크' '카레스테이크'
 '목살오븐구이' '목살스테이크샐러드' '목살짜장스테이크' '목살꼬치구이' '돼지고기콩나물찌개' '쌈장주물럭' '돼지고기목살스테이크'
 '오리엔탈비빔쌀국수' '양념갈비' '돼지목살카레' '고기완자전' '목살필라프' '돼지고추장찌개' '끈기덮밥' '고추장목살구이'
 '묵은지돼지김치찜' '된장돼지구이' '돼지고기묵은지쌈' '묵은지목살찜' '돼지목살스테이크' '목살볶음' '소세지김치찌개'
 '돼지목살끈기덮밥' '목살제육

In [ ]:
# Take user input for the desired menu
desired_menu = input("Enter the desired menu name: ")

# Find and display the ingredients for the desired menu
desired_menu_data = recipe_4[recipe_4['CKG_NM'].str.contains(desired_menu, case=False)]['CKG_MTRL_CN'].iloc[0]

# Split the data into ingredients and instructions using the pipe character '|'
ingredients, instructions = map(str.strip, desired_menu_data.split('|', 1))

print(f"\nIngredients for {desired_menu}:")
print(ingredients)

print("\nInstructions:")
print(instructions)

Enter the desired menu name: 국물불고기

Ingredients for 국물불고기:
[재료] 국간장 3큰술

Instructions:
간장 3큰술| 청주 3큰술| 참기름 1큰술| 올리고당 1큰술| 매실청 1큰술| 다진마늘 1큰술| 후추 1/2작은술| 소고기 600g| 당면 100g| 다시마 멸치육수 3컵| 버섯 100g| 대파 2대| 숙주 1줌| 소금 약간| 후추 약간


In [ ]:
from konlpy.tag import Okt
from tqdm import tqdm

# Assuming you have already defined 'stopwords'
stopwords = ['|']

# 형태소 분석기 OKT를 사용한 토큰화 작업 (다소 시간 소요)
okt = Okt()

tokenized_data = []
for sentence in tqdm(recipe_4['CKG_MTRL_CN']):
    tokenized_sentence = okt.morphs(sentence, stem=True)  # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if word not in stopwords]  # 불용어 제거
    tokenized_data.append(stopwords_removed_sentence)

In [ ]:
# Take user input for the desired menu
desired_menu = input("Enter the desired menu name: ")

# Find the index of the desired menu in the original dataset
menu_indices = recipe_4.index[recipe_4['CKG_NM'].str.contains(desired_menu, case=False)].tolist()

if not menu_indices:
    print(f"No menu found with the name '{desired_menu}'.")
else:
    # Use the first index if there are multiple matches (you can adjust this logic as needed)
    menu_index = menu_indices[0]

    # Retrieve the tokenized data for the desired menu
    if 0 <= menu_index < len(tokenized_data):
        desired_menu_tokens = tokenized_data[menu_index]

        # Join the tokens into strings for both ingredients and instructions
        ingredients = ' '.join(desired_menu_tokens)
        instructions = recipe_4['CKG_MTRL_CN'].iloc[menu_index].split('|', 1)[1].strip()

        print(f"\nIngredients for {desired_menu} (after stopword removal):")
        print(ingredients)

        print("\nInstructions:")
        print(instructions)
    else:
        print(f"Invalid menu index: {menu_index}.")

Enter the desired menu name: 국물불고기
Invalid menu index: 78979.


In [ ]:
# Take user input for the desired menu
desired_menu = input("Enter the desired menu name: ")

# Find and display the ingredients for the desired menu
desired_menu_data = recipe_4[recipe_4['CKG_NM'].str.contains(desired_menu, case=False)]['CKG_MTRL_CN'].iloc[0]

# Tokenize the ingredients using Okt
tokenized_ingredients = okt.morphs(desired_menu_data, stem=True)

# Remove stopwords from the tokenized ingredients
filtered_ingredients = ' '.join([word for word in tokenized_ingredients if word not in stopwords])

print(f"\nIngredients for {desired_menu} (after stopword removal):")
print(ingredients)

print("\nInstructions:")
print(instructions)

Enter the desired menu name: 국물불고기

Ingredients for 국물불고기 (after stopword removal):
[재료] 국간장 3큰술

Instructions:
간장 3큰술| 청주 3큰술| 참기름 1큰술| 올리고당 1큰술| 매실청 1큰술| 다진마늘 1큰술| 후추 1/2작은술| 소고기 600g| 당면 100g| 다시마 멸치육수 3컵| 버섯 100g| 대파 2대| 숙주 1줌| 소금 약간| 후추 약간


In [ ]:
# Take user input for the desired menu
desired_menu = input("Enter the desired menu name: ")

# Find and display the ingredients for the desired menu
desired_menu_data = recipe_4[recipe_4['CKG_NM'].str.contains(desired_menu, case=False)][tokenized_data].iloc[0]

# Tokenize the ingredients using Okt
tokenized_ingredients = okt.morphs(desired_menu_data, stem=True)

# Remove stopwords from the tokenized ingredients
filtered_ingredients = ' '.join([word for word in tokenized_ingredients if word not in stopwords])

print(f"\nIngredients for {desired_menu} (after stopword removal):")
print(ingredients)

print("\nInstructions:")
print(instructions)

Enter the desired menu name: 국물불고기


TypeError: ignored

In [ ]:
# Take user input for the desired menu
desired_menu = input("Enter the desired menu name: ")

# Find the index of the desired menu in the original dataset
menu_indices = recipe_4.index[recipe_4['CKG_NM'].str.contains(desired_menu, case=False)].tolist()

if not menu_indices:
    print(f"No menu found with the name '{desired_menu}'.")
else:
    # Use the first index if there are multiple matches (you can adjust this logic as needed)
    menu_index = menu_indices[0]

    # Check if the index is within the range of the tokenized_data list
    if 0 <= menu_index < len(tokenized_data):
        # Retrieve the tokenized data for the desired menu
        desired_menu_tokens = tokenized_data[menu_index]

        # Join the tokens into strings for both ingredients and instructions
        ingredients = ' '.join(desired_menu_tokens)
        instructions = recipe_4['CKG_MTRL_CN'].iloc[menu_index].split('|', 1)[1].strip()

        print(f"\nIngredients for {desired_menu} (after stopword removal):")
        print(ingredients)

        print("\nInstructions:")
        print(instructions)
    else:
        print(f"Invalid menu index: {menu_index}.")

Enter the desired menu name: 국물불고기
Invalid menu index: 78979.
